## 가계부(입출금 내역)앱의 데이터 처리 부분 단계 
최종 목표는 세 가지

1. 사용자가 입력하거나 CSV로 올린 수입·지출 데이터를 검증하기
2. 월별 통계를 계산하기
3. 결과를 CSV로 저장하기


## 1. 라이브러리와 상수(constant)
* 프로그램 전체에서 공통으로 사용할 컬럼 이름과 데이터 입력 제한 기준을 미리 정해둠
    * COLUMNS   / 날짜, 구분, 항목, 금액  / 필수 컬럼 이름 정의
    * MAX_ROWS  / 5,000                / 거래 내역 최대 행 수 제한
    * MAX_AMOUNT/ 1조 원                / 거래 한 건당 최대 금액 제한
    * MAX_FILE_BYTES/    5MB           / 업로드 파일 최대 크기 제한

In [2]:
from io import BytesIO      # 메모리 안에서 파일처럼 다루는 도구 (업로드된 CSV를 읽거나, 다운로드할 CSV를 만들 때 사용 예정)
from pathlib import Path    # 파일 경로를 다루는 도구 (CSV 저장/불러오기에 사용 예정)

import numpy as np          # 숫자 계산 라이브러리
import pandas as pd         # 표(DataFrame) 데이터를 다루는 라이브러리

In [3]:
# 상수(constant): 프로그램 전체에서 바뀌지 않는 기준값. 관례상 이름을 대문자로 쓴다.
COLUMNS = ['날짜', '구분', '항목', '금액']   # 반드시 있어야 하는 컬럼 4개
MAX_ROWS = 5000                              # 거래 내역은 최대 5,000행
MAX_AMOUNT = 1000000000000                   # 거래 1건당 최대 금액: 1조 원
MAX_FILE_BYTES = 5 * 1024 * 1024 # (약 5.24MB를 십진수 기준으로 환산)

# 설정값 확인용 출력
print("필수 컬럼:", COLUMNS)
print("최대 행 수 :", MAX_ROWS)
print("최대 금액:", f"{MAX_AMOUNT:,}원")              # :, → 천 단위 쉼표 (1,000,000,000,000)
print("최대 CSV 크기:", f"{MAX_FILE_BYTES:,}bytes")

필수 컬럼: ['날짜', '구분', '항목', '금액']
최대 행 수 : 5000
최대 금액: 1,000,000,000,000원
최대 CSV 크기: 5,242,880bytes


## 2. 빈 데이터 만들기: empty_records()
* 데이터가 아직 없더라도 열 이름과 자료형을 미리 지정한 빈 DataFrame을 반환

In [4]:
# 데이터가 하나도 없을 때 쓸 "빈 표"를 만드는 함수
# 행은 0개지만 컬럼 이름과 자료형(dtype)은 미리 정해둔다 → 나중에 데이터를 합칠 때 형식이 맞지 않는 문제를 막음
def empty_records() -> pd.DataFrame:     # -> pd.DataFrame : 이 함수가 DataFrame을 돌려준다는 표시(타입 힌트)
    return pd.DataFrame({
        '날짜': pd.Series(dtype="string"),   # 문자열
        '구분': pd.Series(dtype="string"),   # 문자열 (수입/지출)
        '항목': pd.Series(dtype="string"),   # 문자열 (월급, 식비 등)
        '금액': pd.Series(dtype="int64"),    # 정수
    })

# 테스트: 빈 표가 잘 만들어지는지, 자료형이 맞는지 확인
empty_df = empty_records()
display(empty_df)          # 주피터에서 표를 예쁘게 출력
print(empty_df.dtypes)     # 각 컬럼의 자료형 출력

,날짜,구분,항목,금액


날짜    string
구분    string
항목    string
금액     int64
dtype: object


## 3. 데이터 검증 : validate_records()
* 입출금 데이터가 정해진 규첵에 맞는지 검사하고, 문제가 없으면 데이터를 정리해서 반환하는 함수이다. 
* 검증규칙:
    * 필수 열이 모두 있어야 함
    * 최대 5000행까지 허용
    * 날짜는 실제 날짜인 'YYYY-MM-DD' 형식이며 1900~2100년 범위
    * 구분은 수입 또는 지출 이어야 함
    * 항목은 1~40글자여야함
    * 금액은 1원부터 1조 원까지 양의 정수로 가능
    * 잘못된 행이 있으면 삭제하지 않고 'ValueError'를 발생

In [5]:
# 입출금 데이터가 규칙에 맞는지 검사하고, 깨끗하게 정리한 표를 돌려주는 함수
# 규칙에 어긋나면 ValueError(에러)를 발생시켜 잘못된 데이터가 다음 단계로 넘어가지 않게 막는다
def validate_records(frame:pd.DataFrame) -> pd.DataFrame:

    # print(frame.columns)
    # print(frame.columns.duplicated())   # 컬럼마다 중복된 항목이 있는지 검증

    # 1) 중복 컬럼 확인
    if frame.columns.duplicated().any():  # 중복된 항목이 하나라도 있는지 확인, 있으면 True
        raise ValueError("동일한 컬럼 이름이 중복되었습니다.")

    # 2) 필수 컬럼 확인
    # 리스트 컴프리헨션: COLUMNS 중에서 frame에 없는 이름만 골라 리스트로 만든다
    missing = [name for name in COLUMNS if name not in frame.columns]
    print(missing)
    if missing:   # 빈 리스트면 False, 하나라도 있으면 True
        raise ValueError("필수 컬럼이 없습니다: " + ", " .join(missing))   # ['금액'] → "금액"

    # 3) 최대 행 개수 확인
    if len(frame) > MAX_ROWS:
        raise ValueError(f"내역은 최대{MAX_ROWS:,}행까지 입력할 수 있습니다.")

    # 4) 데이터가 0행이면 더 검사할 게 없으므로 빈 표를 바로 반환
    if frame.empty:
        return empty_records()

    # 5) 필요한 컬럼만 추출
    # .copy() : 원본 frame을 건드리지 않도록 복사본으로 작업
    clean = frame[COLUMNS].copy().reset_index(drop=True)  #drop=True : 기존 인덱스를 새로운 컬럼으로 지정하지 않음

    # 6) 컬럼이 string일 경우 공백 제거
    for name in ["날짜",'구분', '항목']:
        clean[name] = clean[name].astype("string").str.strip()   # 문자열로 바꾼 뒤 앞뒤 공백 제거 (" 월급 " → "월급")


    # 7) 입출금 데이터의 날짜가 올바른 형식인지, 실제 존재하는 날짜인지, 허용된 기간에 포함되는지 검사
    date_text = clean["날짜"]
    # 문자열 → 날짜로 변환. errors="coerce" : 변환할 수 없는 값은 에러 대신 NaT(빈 날짜)로 바꿈
    dates = pd.to_datetime(date_text, format="%Y-%m-%d", errors="coerce")
    # 세 조건을 &(그리고)로 묶어서, 행마다 True(정상)/False(오류)를 담은 Series를 만든다
    valid_dates = (
        date_text.str.fullmatch(r"\d{4}-\d{2}-\d{2}", na=False)  #날짜 문자열이 'YYYY-MM-DD' , na=False : 값이 결측치면 False
        & dates.notna()            # 결측치면 False, 2026-02-29면 False
        & dates.between("1900-01-01","2100-12-31") # 허용된 날짜 범위 검사
    )

    valid_kind = clean["구분"].isin(['수입','지출'])
    #항목이 비어있지 않고, 글자 수가 1~40자 인지를 검사
    valid_category = clean['항목'].notna() & clean["항목"].str.len().between(1,40)
    #금액을 문자열로 변환하고 공백 제거
    raw_amount = clean['금액'].astype("string").str.strip()
    #amount문자열을 숫자로 변환
    amount = pd.to_numeric(raw_amount, errors="coerce")
    #금액이 올바른지 검사(4가지 검사)
    valid_amount = (
        amount.notna() & np.isfinite(amount) # 금액 결측치 확인 & 숫자로 변환할 수 없는 값으로 필터링
        & amount.between(1, MAX_AMOUNT) & amount.mod(1).eq(0) # 금액이 허용 범위에 있는지 & 금액이 소숫점인 값 필터링(1로 나눈 나머지가 0이면 True)
    ).fillna(False)
    
    
    # in 뒤에 튜플(A,B) : A - 검사항목, B - 검사 결과 -> label, mask로 매칭
    for label, mask in[
        ("날짜: YYYY-MM-DD의 실제 날짜 (1900~2100년)",valid_dates),
        ("구분: 수입 또는 지출",valid_kind),
        ("항목: 1~40글자의 이름", valid_category),
        ("금액: 1원~1조원 양의 정수",valid_amount),
    ]:
        # 결과가 하나라도 False이면 실행
        # valid_dates = [True, True, False, True, True] -> 실행
        if not mask.all(): 
            # 날짜 검사해서 잘못된 행 찾기, 최대 10개까지의 문자열 목록으로 저장
            rows= [str(i + 1) for i in clean. index[~mask ] [ : 10] ]
            raise ValueError(f"{label}를 확인하세요. 잘못된 데이터 행: {', '.join(rows)}")
        
    clean['날짜'] = dates.dt.strftime("%Y-%m-%d").astype("string")
    clean['금액'] = amount.astype("int64")
    return clean

In [6]:
# test code
# import pandas as pd

# clean = pd.DataFrame({
#     "구분": ["수입", "지출", "저축", "지출", "저축", "수입"]
# })

# mask = clean["구분"].isin(["수입", "지출"])

# print(clean)
# print(mask)
# print()
# print(~mask)

### 3-1. 정상 데이터 검증하기
문자열 앞뒤 공백을 제거하고, 금액을 정수형으로 변환하는지 확인

In [7]:
# validate_records() 테스트용 샘플 데이터
sample = pd.DataFrame({
    "날짜": ["2026-01-05", "2026-01-08", "2026-02-03", "2026-02-10"],
    "구분": ["수입", "지출", "수입", "지출"],
    "항목": [" 월급 ", "식비", "월급", "교통비"],   # " 월급 " : 공백 제거가 잘 되는지 확인용
    "금액": [3_000_000, "12000", 3_000_000, 50000],   # "12000" : 문자열 금액이 정수로 바뀌는지 확인용 (3_000_000 = 3000000, _는 읽기 쉽게 쓰는 구분자)
})

clean = validate_records(sample)   # 검증 실행
display(clean)
print("\n자료형:")
print(clean.dtypes)

[]


,날짜,구분,항목,금액
0,2026-01-05,수입,월급,3000000
1,2026-01-08,지출,식비,12000
2,2026-02-03,수입,월급,3000000
3,2026-02-10,지출,교통비,50000



자료형:
날짜    string
구분    string
항목    string
금액     int64
dtype: object


In [8]:
# #test code
# 날짜 검사 로직만 따로 떼어 테스트하는 셀 (주석을 풀고 실행하면 결과 확인 가능)
# import pandas as pd

# clean = pd.DataFrame({
#     "날짜": [
#         "2026-01-05",   # 정상 → True
#         "2026-02-30",   # 없는 날짜(2월 30일) → False
#         "2026/03/10",   # 형식이 다름(/ 사용) → False
#         "날짜오류",      # 날짜가 아님 → False
#         "1899-12-31"    # 허용 범위(1900~2100) 밖 → False
#     ]
# })

# date_text = clean["날짜"]

# # 변환할 수 없는 값은 NaT(빈 날짜)가 된다
# dates = pd.to_datetime(
#     date_text,
#     format="%Y-%m-%d",
#     errors="coerce"
# )

# valid_dates = (
#     date_text.str.fullmatch(r"\d{4}-\d{2}-\d{2}", na=False)   # 1) 'YYYY-MM-DD' 모양인가?
#     & dates.notna()                                              # 2) 실제 존재하는 날짜인가?
#     & dates.between("1900-01-01", "2100-12-31")                  # 3) 허용 범위 안인가?
# )

# print(valid_dates)   # 결과: True, False, False, False, False

### 3-2. 잘못된 데이터 시험하기
* 날짜 형식 오류, 잘못된 구분, 0원 금액을 각각 검증

In [9]:
# 일부러 잘못 만든 데이터 3종류 (각각 1행짜리 표)
# 딕셔너리: { "테스트 이름": 잘못된 DataFrame }
bad_examples = {
    "날짜 오류": pd.DataFrame([["2026/01/05", "수입", "월급", 1000]], columns=COLUMNS),   # 날짜에 / 사용
    "구분 오류": pd.DataFrame([["2026-01-05", "환불", "기타", 1000]], columns=COLUMNS),   # 구분이 수입/지출이 아님
    "금액 오류": pd.DataFrame([["2026-01-05", "지출", "식비", 0]], columns=COLUMNS),      # 금액이 0원 (1원 이상이어야 함)
}

# .items() : 딕셔너리의 (키, 값)을 하나씩 꺼낸다
for test_name, bad_df in bad_examples.items():
    try:
        validate_records(bad_df)          # 검증 실행 → 잘못된 데이터이므로 ValueError가 나야 정상
    except ValueError as e:               # 에러가 나면 프로그램을 멈추지 않고 여기로 와서
        print(f"[{test_name}] 검증 실패 확인: {e}")   # 어떤 에러였는지 출력 (e = 에러 메시지)
    # 주의: 에러가 안 나면 아무것도 출력되지 않음 → 해당 검사가 아직 구현되지 않았다는 뜻

[]
[날짜 오류] 검증 실패 확인: 날짜: YYYY-MM-DD의 실제 날짜 (1900~2100년)를 확인하세요. 잘못된 데이터 행: 1
[]
[구분 오류] 검증 실패 확인: 구분: 수입 또는 지출를 확인하세요. 잘못된 데이터 행: 1
[]
[금액 오류] 검증 실패 확인: 금액: 1원~1조원 양의 정수를 확인하세요. 잘못된 데이터 행: 1


## 4. 샘플 파일 읽기 : load_sample()

In [10]:
def load_sample() -> pd.DataFrame:
    path = Path.cwd() / "data" / "sample_transactions.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"샘플 파일을 찾을 수 없습니다. {path}\n"
            "data/sample_transactions.csv파일을 준비해 주세요"
        )
    return validate_records(pd.read_csv(path, encoding="utf-8-sig"))

try:
    loaded = load_sample()
    display(loaded.head())
except FileNotFoundError as e:
    print(e)

샘플 파일을 찾을 수 없습니다. c:\Project_U\01. 프로그래밍 기초\50. Project\00. 나의 입출금 시스템 만들기\data\sample_transactions.csv
data/sample_transactions.csv파일을 준비해 주세요


## 5. CSV 바이트 읽기: read_csv_bytes()
* 파일 업로드 위젯에서 받은 bytes 데이터처럼, 메모리의 CSV 내용을 읽는 함수
* UTF-8 BOM과 CP949 인코딩
* 5MB 초과 파일은 읽지 않음

In [12]:
def read_csv_bytes(content: bytes) -> pd.DataFrame:
    if len(content) > MAX_FILE_BYTES:
        raise ValueError("CSV 파일은 최대 5MB까지 사용할 수 있습니다.")
    if not content:
        raise ValueError("CSV 파일이 비어 있습니다.")
    
    for encoding in ["utf-8-sig", "cp949"]:
        try:
            frame = pd.read_csv(BytesIO(content), encoding=encoding, nrows=MAX_ROWS + 1)
            return validate_records (frame)
        except UnicodeDecodeError:
            continue
        except (pd.errors.EmptyDataError, pd.errors.ParserError) as exc:
            raise ValueError("CSV 형식이 올바르지 않습니다. 쉼표 구분 파일을 사용하세요.") from exc
    raise ValueError("파일 인코딩을 읽을 수 없습니다. UTF-8또는 CP949 CSV로 저장하세요")

csv_content = sample.to_csv(index=False).encode("utf-8-sig")
read_back = read_csv_bytes(csv_content)
display(read_back)

[]


,날짜,구분,항목,금액
0,2026-01-05,수입,월급,3000000
1,2026-01-08,지출,식비,12000
2,2026-02-03,수입,월급,3000000
3,2026-02-10,지출,교통비,50000
